# Trajectories of Change Quickstart

This notebook shows the basic package flow: load the example dataset (two Parquet files), run one metric for one author, plot it, then run and plot the consolidated multimetric overview for several authors. The final sections explain what correct output looks like and how to switch to your own data.

## Install

In Colab, this installs the package with plotting extras; in a local environment where the package is already importable, the cell only prints the version. The pin matches the release this notebook ships with — remove it to get the latest version.

In [ ]:
try:
    import trajectories_of_change
except ImportError:
    %pip install "trajectories-of-change[plotting]==0.2.0"
    import trajectories_of_change

print(f"trajectories_of_change {trajectories_of_change.__version__}")

## Load The Example Bundle

The bundled synthetic oracle has prepared `publications.parquet`, `references.parquet`, and provenance sidecars. Outside the repo, the notebook downloads the same files from GitHub.


In [ ]:
from pathlib import Path
import urllib.request

from trajectories_of_change import load_dataset_bundle, run_metric, run_metrics
from trajectories_of_change.defaults import DEFAULT_TOP_K_KLD_TERMS
from trajectories_of_change.plotting import plot_metric, plot_multimetric

repo_sample_dir = Path("examples/data")
sample_files = [
    "publications.parquet",
    "references.parquet",
    "dataset_manifest.json",
    "run_summary.yaml",
    "config_used.yaml",
]

if repo_sample_dir.exists():
    data_dir = repo_sample_dir
else:
    data_dir = Path("data/example")
    data_dir.mkdir(parents=True, exist_ok=True)
    base_url = "https://raw.githubusercontent.com/raphschlatt/Trajectories_of_Change/v0.2.0/examples/data"
    for name in sample_files:
        target = data_dir / name
        if not target.exists():
            urllib.request.urlretrieve(f"{base_url}/{name}", target)

publications_path = data_dir / "publications.parquet"
references_path = data_dir / "references.parquet"

bundle = load_dataset_bundle(publications_path, references_path, auto_discover_sidecars=True)
print(f"publications: {len(bundle.publications)}")
print(f"references: {len(bundle.references)}")
print(bundle.validation.metric_availability)
if bundle.validation.warnings:
    print("warnings:", bundle.validation.warnings)


## One Metric + Plot

`target_author_uid` is the normal selector. Replace it with an author UID from your own prepared bundle. `skip_incomplete_slices=False` keeps a trailing shortened time window instead of dropping it (the default drops incomplete final windows).

In [ ]:
target_uid = "uid:stable_vocab_distinct"

identity = run_metric(
    bundle,
    metric="citation_identity",
    target_author_uid=target_uid,
    skip_incomplete_slices=False,
    top_k_kld_terms=DEFAULT_TOP_K_KLD_TERMS,
)
plot_metric(identity, export_dir="outputs/author/citation_identity", format="html", show=True)
identity.sync.head()


## Multimetric Example + Plot

`run_metrics` computes all four measures for a set of authors and returns one row per author.

In [ ]:
target_uids = [
    "uid:field_like",
    "uid:stable_vocab_distinct",
    "uid:spiky_vocab_distinct",
    "uid:citation_distinct",
    "uid:density_shift",
    "uid:correlated_distinct",
    "uid:converging_distinct",
    "uid:geometry_trap",
]

metrics_df = run_metrics(
    bundle,
    targets=target_uids,
    select_by="uid",
    skip_incomplete_slices=False,
    top_k_kld_terms=DEFAULT_TOP_K_KLD_TERMS,
    show_progress=False,
)
plot_multimetric(metrics_df, export_dir="outputs/top", format="html", show=True)
metrics_df[[
    "author",
    "author_uid",
    "density_neglog_level",
    "vocab_kld_all_level",
    "cocit_kld_all_level",
    "ref_vocab_kld_all_level",
]].round(4)


## What Should I See?

The bundled example data is synthetic with known expected behavior (an "oracle"): each target author encodes a defined pattern, so you can check the output above against expectations:

- `uid:stable_vocab_distinct` and `uid:correlated_distinct`: clearly elevated `vocab_kld_all_level` (Own Vocabulary divergence).
- `uid:spiky_vocab_distinct`: vocabulary divergence concentrated in spikes rather than a stable elevated level.
- `uid:citation_distinct` and `uid:correlated_distinct`: elevated `cocit_kld_all_level` (Citation Identity divergence).
- `uid:density_shift`: an increasing `density_neglog` trajectory — the author moves into sparser field regions over time.
- `uid:geometry_trap`: near the geometric center of the 2D map yet not in a dense region — density is not geometric centrality.
- `uid:converging_distinct`: a negative KLD slope — the author converges toward the field.
- `uid:field_like`: the baseline; all divergence measures stay low.

If the table and plots match this pattern, your installation computes correctly. The same expectations are asserted by the package's test suite against `examples/data/dataset_manifest.json`.

## CLI Parity

The CLI exposes the same simple paths:

```bash
toc metric citation_identity publications.parquet references.parquet --target-author-uid AUTHOR_UID --out-dir outputs/metric
toc plot metric outputs/metric --out-dir outputs/metric/figures
toc metrics publications.parquet references.parquet --target AUTHOR_UID --out outputs/metrics.parquet
toc plot multimetric outputs/metrics.parquet --out-dir outputs/plots
```


## Use Your Own Data

Swap the example files for your own prepared bundle:

1. Bring two Parquet files that satisfy the data contract: `publications.parquet` (`Bibcode`, `Year`, `Author`, `References`; plus `tokens`, `embedding_2d_x`/`embedding_2d_y`, and `author_uids` for the full metric set) and `references.parquet` (`Bibcode`, `Author`).
2. For raw exports (for example from ADS, with author-name disambiguation applied), run `toc prepare raw_publications.parquet raw_references.parquet --out-dir prepared/` once; it normalizes IDs, deduplicates, and cleans the author-identity layer.
3. Replace the paths in the load cell above with your prepared files and re-run the notebook.
4. Find target UIDs with `run_metrics(bundle, top_n=20)` — the result lists each selected `author_uid` — or inspect `bundle.publications["author_uids"]` directly.

Full input format: [docs/data_contract.md](https://github.com/raphschlatt/Trajectories_of_Change/blob/main/docs/data_contract.md)